# Simple AI Agent on Google Colab

This notebook contains all the code to run the Simple AI Agent web application in a Google Colab environment. Follow the steps below to get started.

## Step 1: Install Dependencies

First, we need to install the necessary Python packages. We'll install `flask` for the web framework and `flask-ngrok` to expose our local web server to the internet.

In [ ]:
!pip install flask flask-ngrok

## Step 2: Create the Project Structure

Next, we'll create the necessary directories and files for our project. This includes the `tools` and `templates` directories, as well as all the Python and HTML files.

In [ ]:
import os

os.makedirs('tools', exist_ok=True)
os.makedirs('templates', exist_ok=True)

In [ ]:
%%writefile tools/search_tool.py
import os

def search(query):
    """Searches for files in the current directory that contain the query."""
    results = []
    for filename in os.listdir("."):
        if query in filename:
            results.append(filename)
    return results if results else "No matching files found."


In [ ]:
%%writefile tools/calculator_tool.py
import operator

def calculate(expression):
    """Calculates the result of a simple arithmetic expression."""
    ops = {
        "+": operator.add,
        "-": operator.sub,
        "*": operator.mul,
        "/": operator.truediv,
    }
    parts = expression.split()
    if len(parts) != 3:
        return "Invalid expression"
    num1, op, num2 = parts
    if op not in ops:
        return f"Unknown operator: {op}"
    try:
        return ops[op](float(num1), float(num2))
    except ValueError:
        return "Invalid numbers"


In [ ]:
%%writefile tools/file_system_tool.py
import os

def list_files(path="."):
    """Lists files in a directory, preventing directory traversal."""
    try:
        # Get the absolute path of the intended directory
        base_dir = os.path.abspath(os.getcwd())
        requested_path = os.path.abspath(os.path.join(base_dir, path))

        # Check if the requested path is within the base directory
        if not requested_path.startswith(base_dir):
            return ["Error: Directory traversal is not allowed."]

        return os.listdir(requested_path)
    except FileNotFoundError:
        return ["Error: Directory not found."]
    except Exception as e:
        return [f"An error occurred: {e}"]


In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Simple AI Agent</title>
</head>
<body>
    <h1>Simple AI Agent</h1>
    <input type="text" id="command" placeholder="Enter command">
    <button onclick="runAgent()">Run</button>
    <pre id="result"></pre>

    <script>
        async function runAgent() {
            const command = document.getElementById("command").value;
            const response = await fetch("/agent", {
                method: "POST",
                headers: {
                    "Content-Type": "application/json"
                },
                body: JSON.stringify({ command })
            });
            const data = await response.json();
            document.getElementById("result").innerText = JSON.stringify(data.result, null, 2);
        }
    </script>
</body>
</html>


In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify, render_template
from flask_ngrok import run_with_ngrok
from tools.search_tool import search
from tools.calculator_tool import calculate
from tools.file_system_tool import list_files

app = Flask(__name__)
run_with_ngrok(app)  # Start ngrok when app is run

class Agent:
    def __init__(self):
        self.tools = {
            "search": search,
            "calculate": calculate,
            "list_files": list_files,
        }

    def run(self, command):
        parts = command.split()
        tool_name = parts[0]
        if tool_name in self.tools:
            arg = " ".join(parts[1:])
            if tool_name == "list_files" and not arg:
                return self.tools[tool_name]()
            return self.tools[tool_name](arg)
        else:
            return f"Unknown tool: {tool_name}"

agent = Agent()

@app.route("/")
def index():
    return render_template("index.html")

@app.route("/agent", methods=["POST"])
def run_agent():
    command = request.json.get("command")
    if not command:
        return jsonify({"error": "Command not provided"}), 400
    result = agent.run(command)
    return jsonify({"result": result})

if __name__ == "__main__":
    app.run()


## Step 3: Run the Application

Now that all the files are in place, we can run the Flask application. The output will include a public URL (from ngrok) that you can use to access the web interface.

In [ ]:
!python app.py